In [25]:
  import sys
  !{sys.executable} -m pip install numpy
  import sys
  !{sys.executable} -m pip install python-sat


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /home/yideun/anaconda3/bin/bin/python -m pip install --upgrade pip
  Obtaining dependency information for python-sat from https://files.pythonhosted.org/packages/46/43/99abfeff02c3fc02dff54bfbe6c6dc5f1d2d5c73746e93c124971cfae0dc/python_sat-1.9.dev2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 22.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: /home/yideun/anaconda3/bin/bin/python -m pip install --upgrade pip


In [26]:
import numpy as np
from itertools import product
import random
import sys
from pysat.solvers import Minisat22

np.set_printoptions(
    threshold=sys.maxsize, # 전체 출력
    linewidth=150,        # 한 줄 길이를 넉넉하게
    precision=3,          # 소수점 3자리까지
    suppress=True         # 0.000001을 0.으로 표시
)

In [27]:
coeff_boundary = 1

## gen_random_qubo
n x n 크기의 랜덤 QUBO 생성

In [40]:
# n x n 크기의 랜덤 qubo를 생성
# 각 element의 범위는 -coeff_boundary ~ coeff_boundary

def gen_random_qubo(n):
    random_qubo = np.random.uniform(-coeff_boundary, coeff_boundary, (n, n))
    return np.triu(random_qubo)  # [수정] 상삼각만 유지, 하삼각은 0

In [41]:
print(gen_random_qubo(5))

[[ 0.424  0.8   -0.807  0.39   0.415]
 [ 0.    -0.908 -0.02  -0.362  0.715]
 [ 0.     0.     0.691  0.256  0.346]
 [ 0.     0.     0.    -0.83   0.252]
 [ 0.     0.     0.     0.     0.19 ]]


## find_opt_brute_force
brute force로 최적해, 최적 값, 축퇴도 탐색

In [30]:
# matrix mat를 입력 받아, 최적해, 최적 값, 축퇴도를 출력
# [수정] num_degenerate 반환 추가 — brute force로 동일 에너지 상태 개수를 세서 유일성 검증
def find_opt_brute_force(mat, debug=False):
    n = mat.shape[0]

    best_x = None
    best_val = float('inf')
    num_degenerate = 0  # [수정] 축퇴도 카운트 추가

    for bits in product([0,1], repeat=n):
        x = np.array(bits)
        cur_val = x@mat@x

        if cur_val < best_val - 1e-12:
            best_val = cur_val
            best_x = x
            num_degenerate = 1
        elif abs(cur_val - best_val) < 1e-12:
            num_degenerate += 1

    if debug: print("opt_x:", best_x, "\nopt_val:", best_val, "\ndegeneracy:", num_degenerate)
        
    return best_x, best_val, num_degenerate

## gen_concatenated_random_qubo
subgraph 분할 → concat하여 큰 QUBO 생성

In [31]:
# n x n 크기의 qubo를 min_sub_graph_size ~ max_sub_graph_size 크기의 행렬들을 concat하여 생성
# 최종 matrix와 optimum을 출력
def gen_concatenated_random_qubo(n, min_sub_graph_size, max_sub_graph_size, debug=False):
    cur_size = 0
    mat = np.zeros((n,n)) # zero qubo 행렬 초기화
    opt = np.zeros(n) # zero 최적해 target 벡터 초기화
    while cur_size < n:
        cur_n = min(random.randint(min_sub_graph_size, max_sub_graph_size), n-cur_size)
        cur_mat = gen_random_qubo(cur_n)
        cur_opt, _, _ = find_opt_brute_force(cur_mat)

        mat[cur_size: cur_size+cur_n, cur_size: cur_size+cur_n] = cur_mat
        opt[cur_size: cur_size+cur_n] = cur_opt

        cur_size += cur_n

        if debug: print(mat)

    return mat, opt

## posiform_planting
posiform planting (MiniSat uniqueness 검증)

In [32]:
# a: alpha
# posiform planting — MiniSat으로 유일성 확보될 때까지 clause 추가
# [수정] 논문(Hahn 2023, Section 2.3) 방식:
#   - 3개 wrong tuple 중 1개만 랜덤 선택하여 추가
# [수정] MiniSat으로 2-SAT uniqueness 검증:
#   - 유일해질 때까지 clause를 계속 추가 (2-SAT phase transition은 O(n))
#   - t 파라미터 제거 — 상한 10*n으로 충분
# [수정] 상삼각 행렬 형식 유지:
#   - off-diagonal 업데이트 시 mat[min(i,j)][max(i,j)]에 접근
def posiform_planting(mat, opt, a):
    n = mat.shape[0]
    all_tuples = [(0, 0), (0, 1), (1, 0), (1, 1)]

    # CNF (Conjunctive Normal Form) clauses 리스트
    # clause = "이 조합은 안 된다"는 배제 조건 (OR of 2 literals)
    #   예: wrong tuple (0,0) 배제 → (xi=1) ∨ (xj=1) → [i+1, j+1]
    # CNF = 모든 clause의 AND → MiniSat에 넘겨서 유일성 검증에 사용
    # (Hahn 2023, Section 2.2: "if the 2-SAT problem has a unique solution,
    #  so does the corresponding posiform")
    clauses_cnf = []

    # 상한: 2-SAT phase transition은 O(n)이므로 10*n이면 충분
    max_clauses = 10 * n

    def add_posiform_term(i, j):
        """pair (i,j)에 대해 wrong tuple 1개를 랜덤 선택하여 posiform 항 + CNF clause 추가"""
        target_tuple = (int(opt[i]), int(opt[j]))

        # target을 제외한 3개 wrong tuple
        wrong_tuples = []
        for t in all_tuples:
            if t != target_tuple:
                wrong_tuples.append(t)

        wi, wj = random.choice(wrong_tuples) # wrong_tuples에서 하나 선택

        # [수정] 상삼각 형식 유지: off-diagonal은 항상 mat[작은][큰]에 접근
        lo, hi = min(i, j), max(i, j)

        # QUBO 행렬 업데이트
        if wi == 0 and wj == 0:   # (1-xi)(1-xj) * a
            mat[i][i] -= a
            mat[j][j] -= a
            mat[lo][hi] += a      # [수정] mat[i][j] → mat[lo][hi]
        elif wi == 0 and wj == 1: # (1-xi)xj * a
            mat[j][j] += a
            mat[lo][hi] -= a      # [수정] mat[i][j] → mat[lo][hi]
        elif wi == 1 and wj == 0: # xi(1-xj) * a
            mat[i][i] += a
            mat[lo][hi] -= a      # [수정] mat[i][j] → mat[lo][hi]
        else:                     # xixj * a
            mat[lo][hi] += a      # [수정] mat[i][j] → mat[lo][hi]

        # CNF clause 추가: wrong tuple (wi,wj) 배제
        #   wi=0 배제 → xi=1이어야 → positive literal (i+1)
        #   wi=1 배제 → xi=0이어야 → negative literal -(i+1)
        lit_i = (i + 1) if wi == 0 else -(i + 1)
        lit_j = (j + 1) if wj == 0 else -(j + 1)
        clauses_cnf.append([lit_i, lit_j])

    def check_uniqueness():
        """MiniSat으로 target 외 다른 해가 있는지 확인."""
        with Minisat22() as solver:
            for clause in clauses_cnf:
                solver.add_clause(clause)

            # target 차단: "적어도 하나의 변수가 target과 달라야 한다"
            blocking = []
            for i in range(n):
                if int(opt[i]) == 1:
                    blocking.append(-(i + 1))
                else:
                    blocking.append(i + 1)

            solver.add_clause(blocking)
            return not solver.solve()

    check_interval = max(1, n // 4)

    for step in range(max_clauses):
        i, j = random.sample(range(n), 2)
        add_posiform_term(i, j)

        if (step + 1) % check_interval == 0:
            if check_uniqueness():
                print(f"  [posiform] {step + 1} clauses로 유일성 확보")
                return

    print(f"  [Warning] {max_clauses} clauses 후에도 유일성 미확보")

## gen_posiform_qubo
random QUBO + posiform planting 결합

In [33]:
def gen_posiform_qubo(n, min_sub_graph_size, max_sub_graph_size, a):
    mat, opt = gen_concatenated_random_qubo(n, min_sub_graph_size, max_sub_graph_size)
    posiform_planting(mat, opt, a)

    return mat, opt, opt@mat@opt

In [34]:
gen_concatenated_random_qubo(10, 5, 5, True) # n, min_sub_graph_size, max_sub_graph_size, debug=False

[[ 0.623 -0.284  0.487  0.842  0.346]
 [-0.947  0.034 -0.795  0.565  0.978]
 [ 0.914  0.633  0.287 -0.636 -0.142]
 [ 0.877 -0.337  0.421  0.354 -0.933]
 [ 0.668  0.722 -0.128 -0.087  0.531]]
[[ 0.623 -0.284  0.487  0.842  0.346  0.     0.     0.     0.     0.   ]
 [ 0.     0.034 -0.795  0.565  0.978  0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.287 -0.636 -0.142  0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.354 -0.933  0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.531  0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.   ]]
[[ 0.794  0.771  0.653  0.21  -0.812]
 [ 0.165  0.714 -0.772 -0.74  -0.781]
 [

(array([[ 0.623, -0.284,  0.487,  0.842,  0.346,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.034, -0.795,  0.565,  0.978,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.287, -0.636, -0.142,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.354, -0.933,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.531,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.794,  0.771,  0.653,  0.21 , -0.812],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.714, -0.772, -0.74 , -0.781],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.666,  0.063, -0.239],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.836,  0.379],
        [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.662]]),
 array([0., 0., 1., 1., 1., 0., 1., 1., 1., 0.]))

In [35]:
# [수정] 검증 루프: 축퇴도(degeneracy) 확인 추가
#   - 기존: 최적해 변경만 확인 (np.array_equal)
#   - 수정: brute force로 ground state 개수도 확인 (deg > 1이면 Warning)
chk = True
cnt = 0
while chk and cnt < 1000:
    cnt += 1
    if(cnt %50 == 0): print(cnt)
    qubo, _ = gen_concatenated_random_qubo(10, 3, 5)
    opt_init, _, _ = find_opt_brute_force(qubo, False)
    
    for _ in range(10):
        posiform_planting(qubo, opt_init, 0.1)
        opt, opt_val, deg = find_opt_brute_force(qubo, False)
        if not np.array_equal(opt_init, opt):
            print("[Error] Optimum is changed!!!")
            chk = False
            break
        if deg > 1:
            print(f"[Warning] Degenerate ground state: {deg} solutions (cnt={cnt})")

print("Done!" if chk else "Failed!")

[[ 0.649  0.91  -0.05  -0.759]
 [-0.226 -0.306  0.629  0.547]
 [-0.449  0.913  0.714 -0.544]
 [ 0.819 -0.942 -0.89  -0.259]]
[[-0.033  0.302  0.532]
 [ 0.696  0.595  0.903]
 [ 0.293 -0.026  0.519]]
[[-0.162 -0.131  0.232]
 [ 0.22   0.422 -0.559]
 [-0.426  0.975 -0.261]]
  [posiform] 84 clauses로 유일성 확보
  [posiform] 52 clauses로 유일성 확보
  [posiform] 24 clauses로 유일성 확보
  [posiform] 68 clauses로 유일성 확보
  [posiform] 56 clauses로 유일성 확보
  [posiform] 44 clauses로 유일성 확보
  [posiform] 36 clauses로 유일성 확보
  [posiform] 60 clauses로 유일성 확보
  [posiform] 46 clauses로 유일성 확보
  [posiform] 26 clauses로 유일성 확보
[[-0.     0.033  0.26  -0.599  0.816]
 [-0.132  0.432 -0.013 -0.205  0.981]
 [ 0.91   0.203  0.114  0.014  0.794]
 [ 0.428  0.127  0.12   0.038 -0.379]
 [ 0.218  0.625 -0.661 -0.566 -0.597]]
[[ 0.882  0.285  0.076 -0.435  0.485]
 [ 0.887 -0.059  0.478  0.111  0.363]
 [ 0.448 -0.771 -0.711  0.019  0.655]
 [-0.496  0.654  0.128 -0.275  0.731]
 [ 0.245  0.727  0.295  0.707  0.166]]
  [posiform] 42 clauses로 유일

In [36]:
n = 50 # 변수 개수
min_sub_graph_size = 5
max_sub_graph_size = 15
a = 0.1 # alpha

qubo, opt_x, opt_val = gen_posiform_qubo(n, min_sub_graph_size, max_sub_graph_size, a)

print("posiform planted qubo:\n", qubo)
print()
print()
print("opt_x:", opt_x)
print(opt_val)

[[-0.905  0.73   0.135 -0.483  0.983  0.344  0.276]
 [-0.469  0.366  0.905 -0.312 -0.954  0.078  0.651]
 [ 0.8   -0.592  0.552  0.007 -0.176 -0.347  0.869]
 [ 0.043  0.512 -0.647  0.014 -0.884  0.515 -0.647]
 [-0.742  0.616 -0.687  0.574 -0.122  0.078  0.968]
 [-0.356 -0.217  0.908 -0.86   0.997 -0.44   0.179]
 [-0.762  0.89   0.786  0.701 -0.132 -0.332  0.651]]
[[ 0.45   0.291 -0.976  0.461 -0.409 -0.463 -0.848]
 [-0.158  0.508  0.324 -0.085 -0.263  0.831 -0.233]
 [-0.658  0.849 -0.618 -0.791 -0.82  -0.359 -0.661]
 [ 0.576 -0.745  0.417 -0.649 -0.742 -0.814 -0.827]
 [-0.675  0.529  0.382 -0.849 -0.484 -0.382 -0.804]
 [-0.43   0.6    0.064  0.235  0.051  0.548  0.841]
 [ 0.312  0.901  0.375  0.068  0.202 -0.263 -0.616]]
[[-0.58   0.292 -0.721 -0.484  0.3   -0.465 -0.087]
 [-0.016 -0.482 -0.967  0.611  0.83   0.537 -0.499]
 [ 0.356 -0.914  0.065  0.616  0.225 -0.916 -0.85 ]
 [-0.129 -0.669 -0.54   0.703 -0.348 -0.415 -0.7  ]
 [-0.086  0.673 -0.068 -0.532  0.214 -0.98   0.914]
 [-0.977  